# MOSIC3 Example: MLP with Budget, Safety, and Fairness Constraints

This notebook demonstrates how to use MOSIC3 for subgroup identification with multiple constraints including:
- **Budget constraints**: Limit total costs
- **Safety constraints**: Bound safety risks in identified subgroups
- **Fairness constraints**: Ensure equitable treatment across sensitive groups

We'll use synthetic data with seed=1 as an example.


## 1. Setup and Imports


In [1]:
import os
os.chdir('..')
import sys
import numpy as np
import torch
import pickle
import random

# Add src to path
sys.path.append('./src')

# Import MOSIC3 components
from MOSIC3 import TwoLayerMLP, MOSIC
from eval_utils import evaluate_result_ContBinary, evaluate_result_ContBinary_DR, evaluate_covariate_balance

# Set random seeds for reproducibility
seed = 1
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

print(f"Using seed: {seed}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Using seed: 1
PyTorch version: 2.3.0+cu121
CUDA available: True


## 2. Configuration Parameters


In [2]:
# Data parameters
GAMMA = 5  # imbalance parameter

# Model parameters
LR = 0.01  # Learning rate
L1_LAMBDA = 0.01  # L1 regularization strength
identifier_type = "mlp"

# Hyperparameters for grid search
HIDDEN_SIZE = 50
BETA = 1e-4

# Constraint parameters
EXPECT_GROUP_SIZE = 0.5  # Expected subgroup size
ALPHA = 0.02  # Overlap threshold

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 3. Load Data

We load the synthetic dataset and pre-computed nuisance functions (IPW weights and DragonNet predictions).


In [3]:
# Load synthetic data
with open(f'data/syn-gamma-{int(GAMMA)}/seed{seed}.pkl', 'rb') as f:
    train_a, train_y, train_Y1, train_Y0, train_X, train_X_raw, \
    test_a, test_y, test_Y1, test_Y0, test_X, test_X_raw = pickle.load(f)

# Load IPW weights
with open(f'data/syn-gamma-{int(GAMMA)}/seed{seed}-ipw.pkl', 'rb') as f:
    ps_train, ipw_train, ps_test, ipw_test = pickle.load(f)

# Load DragonNet predictions
with open(f'data/syn-gamma-{int(GAMMA)}/seed{seed}-dragonnet-output.pkl', 'rb') as f:
    _, pred_Y1_train, pred_Y0_train, pred_Y1_test, pred_Y0_test = pickle.load(f)

print(f"Training set: {train_X.shape[0]} samples, {train_X.shape[1]} features")
print(f"Test set: {test_X.shape[0]} samples")
print(f"Treatment rate (train): {train_a.mean():.3f}")
print(f"Treatment rate (test): {test_a.mean():.3f}")


Training set: 2500 samples, 10 features
Test set: 2500 samples
Treatment rate (train): 0.494
Treatment rate (test): 0.505


## 4. Define Constraints

We construct three types of constraints:
1. **Budget constraint**: Based on feature 2, normalized cost per individual
2. **Safety constraint**: Sigmoid function of the last feature
3. **Fairness constraint**: Equal treatment rates across sensitive groups (feature 3)


In [4]:
# Budget constraint: based on feature 2
train_cost = (train_X[:,2] + 5) / 5
test_cost = (test_X[:,2] + 5) / 5
train_budget_limit = 0.5
test_budget_limit = 0.5

# Safety constraint: sigmoid function of last feature
train_safety_risk = 1 / (1 + np.exp(10 * (train_X[:,-1] + 1)))
test_safety_risk = 1 / (1 + np.exp(10 * (test_X[:,-1] + 1)))
train_safety_risk_limit = 0.05
test_safety_risk_limit = 0.05

# Fairness constraint: sensitive attribute is feature 3
train_sensitive_indicator = np.array(train_X[:, 3] > 0.5, dtype=int)
test_sensitive_indicator = np.array(test_X[:, 3] > 0.5, dtype=int)
train_sensitive_ratio = train_sensitive_indicator.mean()
test_sensitive_ratio = test_sensitive_indicator.mean()
train_fairness_diff_limit = 0.1
test_fairness_diff_limit = 0.1

print(f"Budget - Average cost (train): {train_cost.mean():.3f}, (test): {test_cost.mean():.3f}")
print(f"Safety - Average risk (train): {train_safety_risk.mean():.3f}, (test): {test_safety_risk.mean():.3f}")
print(f"Fairness - Sensitive group ratio (train): {train_sensitive_ratio:.3f}, (test): {test_sensitive_ratio:.3f}")

# Constraint normalization flags
extra_constraint_normalize = torch.tensor([True, True, True, False], dtype=torch.bool)

print(f"\nOriginal test set constraints:")
print(f"  Safety risk: {test_safety_risk.mean():.4f} (limit: {test_safety_risk_limit})")
print(f"  Sensitive ratio: {test_sensitive_ratio:.4f}")
print(f"  Total budget needed: {test_cost.sum():.1f} (limit: {test_budget_limit * test_X.shape[0]:.1f})")


Budget - Average cost (train): 1.000, (test): 1.009
Safety - Average risk (train): 0.162, (test): 0.172
Fairness - Sensitive group ratio (train): 0.296, (test): 0.284

Original test set constraints:
  Safety risk: 0.1718 (limit: 0.05)
  Sensitive ratio: 0.2840
  Total budget needed: 2522.2 (limit: 1250.0)


## 5. Train Final Model

For this example, we'll skip the full grid search and use reasonable default parameters to train the model directly.


In [5]:
# Create identifier model
identifier = TwoLayerMLP(input_size=train_X.shape[1], hidden_size=HIDDEN_SIZE)

# Set up training constraints
train_extra_constraint_a = torch.tensor([
    -train_safety_risk_limit, 
    -(train_fairness_diff_limit + train_sensitive_ratio), 
    -train_fairness_diff_limit + train_sensitive_ratio,
    -train_budget_limit
], dtype=torch.float32)

train_extra_constraint_coeffs = torch.tensor(np.vstack([
    train_safety_risk, 
    train_sensitive_indicator, 
    -train_sensitive_indicator,
    train_cost / train_X.shape[0]
]).T, dtype=torch.float32)

# Create and train final model
model_final = MOSIC(
    identifier=identifier,
    identifier_lr=LR,
    lambda_lr=LR,
    beta=BETA,
    l1=L1_LAMBDA,
    expect_group_size=EXPECT_GROUP_SIZE,
    alpha=ALPHA,
    verbose=False,
    device=DEVICE
)

# Train the model
model_final.fit(
    train_X, train_a, train_y, pred_Y0_train, pred_Y1_train, ps_train, 
    epochs=500,  # Reduced for example
    constraint_a=train_extra_constraint_a, 
    constraint_coeffs=train_extra_constraint_coeffs, 
    constraint_normalize=extra_constraint_normalize
)

print("Training completed!")


Training completed!


## 6. Model Evaluation

Evaluate the trained model on the test set and check constraint satisfaction.


In [6]:
# Get predictions
pred_train = model_final.predict(train_X).cpu().numpy().flatten()
pred_test = model_final.predict(test_X).cpu().numpy().flatten()

# Apply threshold to identify subgroup
pred_threshold = 0.5
test_selected_idx = (pred_test > pred_threshold)

print(f"\n=== Test Set Evaluation ===")
print(f"Selected subgroup size: {test_selected_idx.sum()} / {len(test_selected_idx)} ({test_selected_idx.mean():.3f})")

if test_selected_idx.sum() > 0:
    # Treatment effect estimates
    test_gt_cate = test_Y1[test_selected_idx].mean() - test_Y0[test_selected_idx].mean()
    test_ate_iptw = evaluate_result_ContBinary(test_a[test_selected_idx], test_y[test_selected_idx], ipw_test[test_selected_idx])
    test_ate_aiptw = evaluate_result_ContBinary_DR(
        test_a[test_selected_idx], test_y[test_selected_idx], 
        pred_Y1_test[test_selected_idx], pred_Y0_test[test_selected_idx], ipw_test[test_selected_idx]
    )
    
    # Covariate balance
    test_n_unbalance, _, _ = evaluate_covariate_balance(
        test_X[test_selected_idx,:], test_a[test_selected_idx], 
        ipw_test[test_selected_idx], device=DEVICE, smd_threshold=0.2
    )
    
    # Constraint satisfaction
    test_safety = test_safety_risk[test_selected_idx].mean()
    test_ratio_diff = np.abs(test_sensitive_indicator[test_selected_idx].mean() - test_sensitive_ratio)
    test_cost_sum = test_cost[test_selected_idx].sum()
    
    print(f"\n--- Treatment Effects ---")
    print(f"Ground truth CATE: {test_gt_cate:.4f}")
    print(f"IPTW estimate: {test_ate_iptw:.4f}")
    print(f"AIPTW estimate: {test_ate_aiptw:.4f}")
    
    print(f"\n--- Covariate Balance ---")
    print(f"Number of unbalanced covariates: {test_n_unbalance}")
    
    print(f"\n--- Constraint Satisfaction ---")
    print(f"Safety risk: {test_safety:.4f} (limit: {test_safety_risk_limit:.4f}) {'✓' if test_safety <= test_safety_risk_limit else '✗'}")
    print(f"Fairness difference: {test_ratio_diff:.4f} (limit: {test_fairness_diff_limit:.4f}) {'✓' if test_ratio_diff <= test_fairness_diff_limit else '✗'}")
    print(f"Budget cost: {test_cost_sum:.1f} (limit: {test_budget_limit * test_X.shape[0]:.1f}) {'✓' if test_cost_sum <= test_budget_limit * test_X.shape[0] else '✗'}")
    
else:
    print("No samples selected in subgroup!")



=== Test Set Evaluation ===
Selected subgroup size: 1188 / 2500 (0.475)

--- Treatment Effects ---
Ground truth CATE: 0.0659
IPTW estimate: -0.0706
AIPTW estimate: 0.0531

--- Covariate Balance ---
Number of unbalanced covariates: 0

--- Constraint Satisfaction ---
Safety risk: 0.0466 (limit: 0.0500) ✓
Fairness difference: 0.1015 (limit: 0.1000) ✗
Budget cost: 1194.7 (limit: 1250.0) ✓
